In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import math
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen
from scipy.integrate import solve_ivp
import pandas as pd
import glob
import os
from pathlib import Path

In [ ]:


# ---- Параметры -------------------------------------------------------------
FOLDER_WITH_EXCEL = r"./LP_SP"    # папка, где лежат .xlsx
PATTERN = "*.xlsx"                      # при необходимости можно изменить на *.xlsm и т.д.
ADD_SOURCE_COLUMNS = True               # добавить колонки с именем файла/листа
# ---------------------------------------------------------------------------

def read_all_excels(folder, pattern="*.xlsx", add_source_cols=True):
    """
    Читать все Excel-файлы из папки и вернуть один объединённый DataFrame.

    :param folder: Папка с .xlsx
    :param pattern: Глоб-шаблон для поиска файлов
    :param add_source_cols: Добавлять ли столбцы __file__ и __sheet__
    """
    full_pattern = os.path.join(folder, pattern)
    files = glob.glob(full_pattern)

    if not files:
        raise FileNotFoundError(f"По шаблону {full_pattern!r} ничего не найдено")

    frames = []  # сюда будем складывать все DataFrame'ы

    for fname in files:
        # Читаем сразу все листы: вернётся словарь {имя_листа: DataFrame}
        sheets_dict = pd.read_excel(fname, sheet_name=None, engine="openpyxl")

        for sheet_name, df in sheets_dict.items():
            if add_source_cols:
                df = df.copy()  # чтобы не трогать оригинал
                df["file"]  = os.path.basename(fname)
                df["sheet"] = sheet_name
            frames.append(df)

    # Объединяем
    combined = pd.concat(frames, ignore_index=True)
    return combined

In [ ]:
df_all = read_all_excels(FOLDER_WITH_EXCEL)

In [ ]:
df_all['mu'] = [float(x.replace('.xlsx','').replace('mu=','')) for x in df_all['file']]

In [ ]:
df_all.head()

In [ ]:
df_SP = df_all[df_all['sheet'] == 'SP']

In [ ]:
mu_list = list(set(df_SP['mu']))
mu0 = mu_list[1]

In [ ]:
def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt

In [ ]:
from scipy.interpolate import CubicSpline

In [ ]:
mu_list = list(set(df_SP['mu']))
mu_list.sort()
mu_list = np.array(mu_list)
v1_list = []
df_SP_mu = df_SP[df_SP['mu'] == mu_list[0]]
a_etalon = np.array(df_SP_mu['a'])
for mu in mu_list:
    df_SP_mu = df_SP[df_SP['mu'] == mu]
    cs = CubicSpline(np.array(df_SP_mu['a']), np.array(df_SP_mu['v1']))
    v1_corrected = cs(a_etalon)
    v1_list.append(v1_corrected)
    plt.plot(a_etalon,v1_corrected)
v1 = np.array(v1_list)
plt.grid()
plt.xlabel('a')

In [ ]:
mu_list = list(set(df_SP['mu']))
mu_list.sort()
mu_list = np.array(mu_list)
v1_list = []
v2_list = []
T_list = []
df_SP_mu = df_SP[df_SP['mu'] == mu_list[0]]
a_etalon = np.array(df_SP_mu['a'])
for mu in mu_list:
    df_SP_mu = df_SP[df_SP['mu'] == mu]
    
    cs = CubicSpline(np.array(df_SP_mu['a']), np.array(df_SP_mu['v1']))
    v1_corrected = cs(a_etalon)
    v1_list.append(v1_corrected)
    
    cs = CubicSpline(np.array(df_SP_mu['a']), np.array(df_SP_mu['v2']))
    v2_corrected = cs(a_etalon)
    v2_list.append(v2_corrected)
    
    cs = CubicSpline(np.array(df_SP_mu['a']), np.array(df_SP_mu['T']))
    T_corrected = cs(a_etalon)
    T_list.append(T_corrected)
v1 = np.array(v1_list)
v2 = np.array(v2_list)
T = np.array(T_list)

a = np.tile(a_etalon, (v1.shape[0], 1))
mu = np.tile(mu_list, (v1.shape[1], 1)).T

In [ ]:
mu.shape

In [ ]:
# Plot the surface.
X = a
Y = mu
Z = v1
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)
plt.grid()
#plt.legend(loc='upper right')
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$v_x$, безразм.')


In [ ]:
# Plot the surface.
X = a
Y = mu
Z = v2
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)
plt.grid()
#plt.legend(loc='upper right')
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$v_y$, безразм.')


In [ ]:
# Plot the surface.
X = a
Y = mu
Z = T
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)

ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$T$, безразм.')

In [ ]:
import numpy as np

# Зададим размеры
n_mu, n_a = v2.shape          # (10, 838)

# сгладим индексы, чтобы потом удобно выбирать
idx_flat = np.arange(n_mu * n_a).reshape(n_mu, n_a)
# шаги по координатам, если х и m равномерные
da = np.mean(np.diff(a[0]))      # по x берём первую строку
dmu = np.mean(np.diff(mu[:, 0]))   # по m берём первый столбец

dv_dmu, dv_da = np.gradient(v2, dmu, da)      # градиенты
grad_norm = np.hypot(dv_da, dv_dmu)

# берём 500 точек с наибольшим градиентом
k = 500
flat_best = np.argpartition(grad_norm.ravel(), -k)[-k:]  # быстрая сортировка
i_sel, j_sel = np.unravel_index(flat_best, v2.shape)


In [ ]:
# сначала первые производные
dv_dmu, dv_da = np.gradient(v2, dmu, da)
# теперь вторые
d2v_dmu2, d2v_dadmu = np.gradient(dv_dmu, dmu, da)
d2v_dmuda, d2v_da2  = np.gradient(dv_da, dmu, da)

curvature = np.hypot(d2v_da2, d2v_dmu2)          # простая мера кривизны
flat_best = np.argpartition(curvature.ravel(), -k)[-k:]
i_sel, j_sel = np.unravel_index(flat_best, v2.shape)

In [ ]:
# возьмём полином 3×3 по x и m
from numpy.polynomial.polynomial import polyvander2d, polyval2d

deg = 3
V = polyvander2d(a.ravel(), mu.ravel(), [deg, deg])
coef, *_ = np.linalg.lstsq(V, v2.ravel(), rcond=None)
v2_pred = polyval2d(a, mu, coef.reshape(deg+1, deg+1))  # восстановим форму

error = np.abs(v2 - v2_pred)
flat_best = np.argpartition(error.ravel(), -k)[-k:]
i_sel, j_sel = np.unravel_index(flat_best, v2.shape)

In [ ]:
from scipy.ndimage import generic_filter

def local_var(arr):
    return arr.var()

roughness = generic_filter(v2, local_var, size=3, mode='nearest')
flat_best = np.argpartition(roughness.ravel(), -k)[-k:]
i_sel, j_sel = np.unravel_index(flat_best, v2.shape)

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# 1) беру top-2000 по градиенту
biggrad = 2000
flat_big = np.argpartition(grad_norm.ravel(), -biggrad)[-biggrad:]
coords_big = np.column_stack(np.unravel_index(flat_big, v2.shape))  # (N,2)

# 2) кластеризуем эти координаты до k=500 центров
kmeans = MiniBatchKMeans(n_clusters=500, random_state=0).fit(coords_big)
centers = kmeans.cluster_centers_.astype(int)

i_sel, j_sel = centers[:,0], centers[:,1]    # итоговые индексы

In [ ]:
import numpy as np
m = mu
x = a
n_m, n_x = v2.shape
k = 500                       # целевой размер
dm = np.mean(np.diff(m[:, 0]))
dx = np.mean(np.diff(x[0]))
# «важность» = норма градиента (можно заменить на кривизну, ошибку и т.д.)
dv_dm, dv_dx = np.gradient(v2, dm, dx)
imp = np.hypot(dv_dm, dv_dx)          # (10, 838), ненулевая и положительная
imp = (imp - imp.min()) / (imp.ptp() + 1e-12) + 1e-3    # нормировка 0…1

In [ ]:
T

In [ ]:
S = int(np.sqrt(k * n_m / n_x)) or 1     # ≈ число полос по m
T = int(np.sqrt(k * n_x / n_m)) or 1     # ≈ число полос по x
# границы
edges_m = np.linspace(0, n_m, S+1, dtype=int)
edges_x = np.linspace(0, n_x, T+1, dtype=int)

chosen = []
for im0, im1 in zip(edges_m[:-1], edges_m[1:]):
    for ix0, ix1 in zip(edges_x[:-1], edges_x[1:]):
        block = imp[im0:im1, ix0:ix1]
        if block.size:
            # индекс в пределах блока
            loc = np.argmax(block)
            g, h = np.unravel_index(loc, block.shape)
            chosen.append((im0+g, ix0+h))
# если вдруг <500, дополняем важными невзятыми; если >500, случайно убираем
if len(chosen) < k:
    flat = np.argsort(-imp.ravel())      # глобальная сортировка
    for p in flat:
        idx = divmod(p, n_x)
        if idx not in chosen:
            chosen.append(idx)
        if len(chosen) == k:
            break
else:
    np.random.shuffle(chosen)
    chosen = chosen[:k]

i_sel, j_sel = map(np.array, zip(*chosen))

In [ ]:
# Этот способ работает
r_min = np.hypot(n_m, n_x) / np.sqrt(k * np.pi)   # эмпирически ~равномерно

# 1) генерация много-много (3–5k) кандидатов с весами
n_cand = 5 * k
flat = np.random.choice(n_m*n_x, n_cand, replace=False, p=imp.ravel()/imp.sum())
cands = np.column_stack(np.unravel_index(flat, v2.shape)).astype(float)

# 2) Poisson-disk фильтр (O(k^2), здесь маленькие n, поэтому ок)
chosen = []
for p in cands:
    if all(np.linalg.norm(p - q) >= r_min for q in chosen):
        chosen.append(tuple(p.astype(int)))
    if len(chosen) == k:
        break
# если не набрали 500, слегка уменьшаем r_min и продолжаем
while len(chosen) < k:
    r_min *= 0.9
    for p in cands:
        if all(np.linalg.norm(p - q) >= r_min for q in chosen):
            chosen.append(tuple(p.astype(int)))
        if len(chosen) == k:
            break

i_sel, j_sel = map(np.array, zip(*chosen))

In [ ]:
alpha = 2.0
weights = 1 + alpha * imp          # shape = (n_m, n_x)

# массив координат (N,2) для удобства
coords = np.indices(v2.shape).reshape(2, -1).T   # (N,2)
# расстояния будем хранить вектором длиной N
d2 = np.full(coords.shape[0], np.inf)

# 1) начальная точка
start = np.argmax(imp)
picked = [start]

for _ in range(1, k):
    # обновляем минимальное расстояние до набора
    new_pt = coords[picked[-1]]
    diff = coords - new_pt
    d2 = np.minimum(d2, (diff**2).sum(1))
    # целевая функция: w * dist²
    score = weights.ravel() * d2
    nxt = np.argmax(score)
    picked.append(nxt)

inds = np.array(picked)
i_sel, j_sel = coords[inds].T

In [ ]:

# сами подмаскированные данные
v2_sub = v2[i_sel, j_sel]
a_sub = a[i_sel, j_sel]
mu_sub = mu[i_sel, j_sel]
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.scatter(a_sub, mu_sub,  v2_sub)

fig.show()

In [ ]:
mu = mu_list[1]

df_SP_mu = df_SP[df_SP['mu'] == mu]

a = np.array(df_SP_mu['a'])
x1 = np.array(df_SP_mu['x1'])
x2 = np.array(df_SP_mu['x2'])
v1 = np.array(df_SP_mu['v1'])
v2 = np.array(df_SP_mu['v2'])
T = np.array(df_SP_mu['T'])

In [ ]:
fig_man, ax_man = plt.subplots(1)
fig_res, ax_res = plt.subplots(1)
res_list = []
for i in range(0, x1.size, 5):
    #print(i)
    p0 = np.array([x1[i], x2[i], v1[i], v2[i]])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:])
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1[i], res, '.k')
    res_list.append(res)
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
max(mu_list)

In [ ]:
np.max(a)

In [ ]:
np.pi*np.pi/2

In [ ]:
4.928138/np.pi

In [ ]:
L1 =  0.836915131427382
L2 =  1.155682161024677
L3 = -1.005062645331442

#Земля-Луна
mu0 = 1/82
#Солнце-Земля
mu0 = 3.0035e-6

#Солнце-Меркурий
#mu0 = 1.66e-7

#предел
#mu0 = 0.03852089

#def v1_fun(x):
#    y = (-0.31065616*((x + 0.04729554*math.sin(1.2135408*x + 1.6476246))*(5.527562*x + math.cos(1.7415938*x**1.5302258)) - 0.04729554)*math.sin(0.8843217*x + 2.52133)/(5.527562*x + math.cos(1.7415938*x**1.5302258))) #1.4e-04
#    return y
#def v2_fun(x):
#    y = ((x - 0.0032811477*(0.83023274*x)**x)*(x*(-0.076037794*x + 0.15013573*math.sin(x) + 0.11507184*math.cos(x) + 0.20255086) + 0.002156449)/x) #1.1e-04
#    return y
#def T_fun(x):
#    y = ((1.3619989 - 0.056624785*0.061825007**x)*(0.48113042**x + 0.05254563*(x + math.cos(0.37977582*x**2))**0.07269894*math.cos(x - 0.62925476) + 4.5865707)) #2.6e-04
#    return y

def x1_fun(a, mu):
    #y = (-mu + math.sin(a + np.pi*5/6))
    y = math.cos(a + np.pi/3) - mu
    return y
def x2_fun(a, mu):
    #y = (math.sin(a - np.pi*2/3))
    y = -math.sin(a + np.pi/3)
    return y
def v1_fun(a, mu):
    y = (-a*((mu*np.cos(0.22937126**a - np.log(a)) - 0.41244736*np.cos(0.22130254**a + np.log(a)))*np.sin(a + 0.028886221) + 0.26014933*np.cos(1.2741468*a + 5.4069505))) #1.1e-03
    return y
def v2_fun(a, mu):
    y = (a*(-6.557952*mu + 0.26311928*np.exp(1.4665749*a) + 0.415885222275072)*np.exp(-1.4665749*a)*np.cos(0.5028553*a*mu - a + 0.93205124)) #2.7e-03
    #y = (((0.6295174*a - 3.436374)*(mu*(0.26111934*math.sin(a) + 1.22414) - 0.020510118)*math.cos(1.3607086*a - 1.0926718) + math.sin(1.2466944*a))/(3.436374 - 0.6295174*a)) #2.1e-03
    return y
def T_fun(a, mu):
    #y = (mu*(4.928138*math.cos(a - 0.6908796) + 2.9778968732458 + 43.95868/(math.exp(a) + math.sin(88.58396*mu)))*math.cos(math.exp(-83.87106*mu)) + 6.283895) #1.4e-02
    #y = mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + 45.56829193/(np.exp(a) + np.sin(87.19693384*mu)))*np.cos(np.exp(-65.93754813*mu)) + np.pi*2 #1.3e-02
    y = mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + (29/2)*np.pi/(np.exp(a) + np.sin(np.pi*(111/4)*mu)))*np.cos(np.exp(-21*np.pi*mu)) + np.pi*2 #1.3e-02
    return y
    
#x_range = np.linspace(0.005, 4.19, 100)

x_range = np.linspace(0.005, 4.19, 10)

fig_man, ax_man = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i], mu0), x2_fun(x_range[i], mu0), v1_fun(x_range[i], mu0), v2_fun(x_range[i], mu0)])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu0),
                  t_span=(0, T_fun(x_range[i], mu0)),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i], mu0), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], 'k')
    #ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
#ax_man.plot(L3, 0, '*k')
ax_man.plot(x1_fun(0, mu0), x2_fun(0, mu0),'*k')
ax_man.grid()
#ax_res.grid()

ax_man.set_xlabel('x, безразм.', fontsize=11)
ax_man.set_ylabel('y, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'S3_orbits.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')


In [ ]:
x_range = np.linspace(0.005, 4.19, 100)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i], mu0), x2_fun(x_range[i], mu0), v1_fun(x_range[i], mu0), v2_fun(x_range[i], mu0)])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu0),
                  t_span=(0, T_fun(x_range[i], mu0)),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i], mu0), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_res.grid()
ax_res.set_xlabel(r'$x$, безразм.', fontsize=11)
ax_res.set_ylabel(r'Невязка через период, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'S3_err.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:
mu

In [ ]:
err = mu*0
for i in range(mu.shape[0]):
    for j in range(mu.shape[1]):
        #print(i)
        p0 = np.array([x1_fun(a[i, j], mu[i, j]), x2_fun(a[i, j], mu[i, j]), v1_fun(a[i, j], mu[i, j]), v2_fun(a[i, j], mu[i, j])])
        p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu[i, j]),
                      t_span=(0, T_fun(a[i, j], mu[i, j])),
                      y0=p0,
                      method='LSODA',
                      t_eval = np.linspace(0, T_fun(a[i, j], mu[i, j]), 1000),
                      atol=1e-12,
                      rtol=1e-12)
        err[i,j]= np.linalg.norm(p.y[:,-1] - p0)

In [ ]:
# Plot the surface.
X = a
Y = mu
Z = err
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)

ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'Невязка через период, безразм.')

In [ ]:
plt.plot(a, v1)
plt.plot(a, [v1_fun(x) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
# Plot the surface.
X = a
Y = mu
Z = T
Z2 = T_fun(a, mu)
Z_diff = Z - Z2
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)


ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$T$, безразм.')

In [ ]:
#### import numpy as np
import matplotlib.pyplot as plt

# Ваши данные
X = a
Y = mu
Z = T
Z2 = T_fun(a, mu)
Z_diff = Z - Z2

# Вариант без tight_layout (рекомендуется с 3D + inset)
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Основная 3D-поверхность/каркас
ax.plot_wireframe(X, Y, Z, color='C0', linewidth=0.6)
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$T$, безразм.')

# Вложенная 3D-ось через современный API (координаты в долях родительской оси)
axins = ax.inset_axes([0.4, 0.45, 0.36, 0.36], projection='3d', zorder=5)
axins.set_facecolor('none')  # прозрачный прямоугольник оси (не закрашивает то, что под ней)

# Что рисуем: модуль ошибки (или замените на Z_diff для знаковой)
W = np.abs(Z_diff)

# Каркас ошибки
axins.plot_wireframe(
    X, Y, W,
    rcount=40, ccount=40,
    color='C3', linewidth=0.6, alpha=0.9
)

# Синхронизируем угол обзора с главным графиком
axins.view_init(elev=ax.elev, azim=ax.azim)

# Минимизируем визуальный шум
axins.set_title(r'Разница', fontsize=9)
axins.set_zlabel('Δ, безразм.', fontsize=8)
#axins.set_xticks([]); axins.set_yticks([]); axins.set_zticks([])
#axins.set_box_aspect((np.ptp(X), np.ptp(Y), np.ptp(W)))

plt.show()

In [ ]:
#### import numpy as np
import matplotlib.pyplot as plt

# Ваши данные
X = a
Y = mu
Z = v1
Z2 = v1_fun(a, mu)
Z_diff = Z - Z2

# Вариант без tight_layout (рекомендуется с 3D + inset)
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Основная 3D-поверхность/каркас
ax.plot_wireframe(X, Y, Z, color='C0', linewidth=0.6)
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$v_x$, безразм.')

# Вложенная 3D-ось через современный API (координаты в долях родительской оси)
axins = ax.inset_axes([0.1, 0.5, 0.36, 0.36], projection='3d', zorder=5)
axins.set_facecolor('none')  # прозрачный прямоугольник оси (не закрашивает то, что под ней)

# Что рисуем: модуль ошибки (или замените на Z_diff для знаковой)
W = np.abs(Z_diff)

# Каркас ошибки
axins.plot_wireframe(
    X, Y, W,
    rcount=40, ccount=40,
    color='C3', linewidth=0.6, alpha=0.9
)

# Синхронизируем угол обзора с главным графиком
axins.view_init(elev=ax.elev, azim=ax.azim)

# Минимизируем визуальный шум
axins.set_title(r'Разница', fontsize=9)
axins.set_zlabel('Δ, безразм.', fontsize=8)
#axins.set_xticks([]); axins.set_yticks([]); axins.set_zticks([])
#axins.set_box_aspect((np.ptp(X), np.ptp(Y), np.ptp(W)))

plt.show()

In [ ]:
#### import numpy as np
import matplotlib.pyplot as plt

# Ваши данные
X = a
Y = mu
Z = v2
Z2 = v2_fun(a, mu)
Z_diff = Z - Z2

# Вариант без tight_layout (рекомендуется с 3D + inset)
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Основная 3D-поверхность/каркас
ax.plot_wireframe(X, Y, Z, color='C0', linewidth=0.6)
ax.set_xlabel(r'$a$, безразм.')
ax.set_ylabel(r'$\mu$, безразм.')
ax.set_zlabel(r'$v_y$, безразм.')

# Вложенная 3D-ось через современный API (координаты в долях родительской оси)
axins = ax.inset_axes([0.1, 0.25, 0.36, 0.36], projection='3d', zorder=5)
axins.set_facecolor('none')  # прозрачный прямоугольник оси (не закрашивает то, что под ней)

# Что рисуем: модуль ошибки (или замените на Z_diff для знаковой)
W = np.abs(Z_diff)

# Каркас ошибки
axins.plot_wireframe(
    X, Y, W,
    rcount=40, ccount=40,
    color='C3', linewidth=0.6, alpha=0.9
)

# Синхронизируем угол обзора с главным графиком
axins.view_init(elev=ax.elev, azim=ax.azim)

# Минимизируем визуальный шум
axins.set_title(r'Разница', fontsize=9)
axins.set_zlabel('Δ, безразм.', fontsize=8)
#axins.set_xticks([]); axins.set_yticks([]); axins.set_zticks([])
#axins.set_box_aspect((np.ptp(X), np.ptp(Y), np.ptp(W)))

plt.show()

In [ ]:
# Plot the surface.
X = a
Y = mu
Z = v2 - v2_fun(a, mu)
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)


fig.show()

In [ ]:
# Plot the surface.
X = a
Y = mu
Z = (T - T_fun(a, mu))/T
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Plot the surface.
surf = ax.plot_wireframe(X, Y,  Z)


fig.show()

In [ ]:
1/mu

In [ ]:
def sklfun_base(x, a, mu):
    y = mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + 14.5*np.pi/(np.exp(a) + np.sin(np.pi*(111/4)*mu)))*np.cos(np.exp(-21*np.pi*mu)) + np.pi*2 #1.3e-02
    return y

def sklfun(x, a, mu, y):
    return sklfun_base(x, a, mu) - y

x0 = [4.928138,#0
      - np.pi/5,#1
      1,#2
      43.95868, #3
      88.58396,#4
      -83.87106,#5
     ]
true_array = T.flatten()
res_lsq = least_squares(sklfun, x0, args=(a.flatten(), mu.flatten(), true_array))

pred_array = sklfun_base(res_lsq.x, a.flatten(), mu.flatten())
MSE = np.square(np.subtract(true_array,pred_array)).mean() 
RMSE_train = np.sqrt(MSE)
print('Train:',RMSE_train)
display(res_lsq.x)

In [ ]:
87.2073897/np.pi

In [ ]:
111/4

In [ ]:
from fractions import Fraction

In [ ]:

    y = mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + 45.56829193/(np.exp(a) + np.sin(87.19693384*mu)))*np.cos(np.exp(-65.93754813*mu)) + np.pi*2 #1.3e-02

In [ ]:
from fractions import Fraction
print(Fraction(87.2073897/np.pi).limit_denominator(10))

In [ ]:
def v1_fun(a, mu):
    return -a * (mu + 1 / np.pi) * np.sin(2 * np.pi / 7 * a + 4 * np.pi / 5)

In [ ]:
def v2_fun(a, mu):
    y = -(-0.1580214*a + mu -0.32662316 + 0.34295581 *a * mu *np.cos (a) ) * np.cos(a -1.4691385*mu -1.10383876) -0.1277802 #3.8e-03
    return y

In [ ]:
def v2_fun(a, mu):
    x = [-0.157,#0
      -0.331,#1
      0.337,#2
      -1.436, #3
      - 1.108,#4
      - 0.131,#5
      1.1e-3,#6
      3.05,#7,
      2.3e-4,#8
      0.1,#9
      -5.0e-4,#10
     ]
    A0 = -(x[0]*a + mu +x[1] + x[2] *a * mu *np.cos (a) )
    A = A0*(1 +x[6] * np.cos(x[7]*a)+x[8]*np.log(a + x[9])+x[10]*mu*np.sin(a)) #
    phi = a +x[3]*mu +x[4]#+x[6] * np.cos(x[7]*a)
    y = A  * np.cos(phi) + x[5]#+x[6]* mu *np.sin(a)
    return y

In [ ]:
def v2_fun(a, mu):
    x = [-0.12311518, -0.29058251,  0.27659109, -1.55481652, -1.02249145,
       -0.06691507, -0.01309215,  2.52222503,  0.19184952,  0.06276367,
       -0.3870346 ]
    A0 = -(x[0]*a + mu +x[1] + x[2] *a * mu *np.cos (a) )
    A = A0*(1 +x[6] * np.cos(x[7]*a)+x[8]*np.log(a + x[9])+x[10]*mu*np.sin(a)) #
    phi = a +x[3]*mu +x[4]#+x[6] * np.cos(x[7]*a)
    y = A  * np.cos(phi) + x[5]#+x[6]* mu *np.sin(a)
    return y

In [ ]:
def sklfun_base(x, a, mu):
    A = -(x[0]*a + mu +x[1] + x[2] *a * mu *np.cos (a))
    phi = a +x[3]*mu +x[4]
    y = A  * np.cos(phi) + x[5]#+x[6]* mu *np.sin(a)
    return y

def sklfun(x, a, mu, y):
    return sklfun_base(x, a, mu) - y

x0 = [-0.157,#0
      -0.331,#1
      0.337,#2
      -1.436, #3
      - 1.108,#4
      - 0.131,#5
      
      1.1e-3,#6
      3.05,#7,
      2.3e-4,#8
      0.1,#9
      -5.0e-4,#10
      0,#11
      3,#12
     ]
true_array = v1.flatten()
res_lsq = least_squares(sklfun, x0, args=(a.flatten(), mu.flatten(), v1.flatten()))

pred_array = sklfun_base(res_lsq.x, a.flatten(), mu.flatten())
MSE = np.square(np.subtract(true_array,pred_array)).mean() 
RMSE_train = np.sqrt(MSE)
print('Train:',RMSE_train)
display(res_lsq.x)

In [ ]:
def v2_fun(a, mu):
    y = -(-0.1580214*a + mu -0.32662316 + 0.34295581 *a * mu *np.cos (a) ) * np.cos(a -1.4691385*mu -1.10383876) -0.1277802
    return y

In [ ]:
plt.plot(a, v2)
plt.plot(a, [v2_fun(x, mu0) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()

In [ ]:
plt.plot(a, x1)
plt.plot(a, [x1_fun(x, mu) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
plt.plot(a, x2)
plt.plot(a, [x2_fun(x) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
2.0943952/np.pi

In [ ]:
def v2_fun(x):
    y = ((x - 0.0032811477*(0.83023274*x)**x)*(x*(-0.076037794*x + 0.15013573*math.sin(x) + 0.11507184*math.cos(x) + 0.20255086) + 0.002156449)/x) #1.1e-04
    return y

In [ ]:
def sklfun_base(x, t):
    y = (t + x[0]*(x[1]*t)**t)*(t*(x[2]*t + x[3]*np.sin(t) + x[4]*np.cos(t) + x[5]) + x[6])/t #1.1e-04
    return y

def sklfun(x, t, y):
    return sklfun_base(x, t) - y

x0 = [-0.0032811477,#0
      0.83023274,#1
      -0.076037794,#2
      0.15013573, #3
      0.11507184,#4
      0.20255086,#5
      0.002156449,#6
     ]
mu_list = list(set(df_SP['mu']))
mu_list.sort()

x_list = []
for mu in mu_list:
    df_SP_mu = df_SP[df_SP['mu'] == mu]
    
    t_array = np.array(df_SP_mu['a'])
    true_array = np.array(df_SP_mu['v2'])
    
    res_lsq = least_squares(sklfun, x0, args=(t_array, true_array))
    
    pred_array = sklfun_base(res_lsq.x, t_array.reshape(true_array.shape))
    MSE = np.square(np.subtract(true_array,pred_array)).mean() 
    RMSE_train = np.sqrt(MSE)
    print('Train:',RMSE_train)
    x_list.append(res_lsq.x)

In [ ]:
plt.plot(mu_list, x_list)
plt.grid()

In [ ]:
model = PySRRegressor(
        niterations=500,  # < Increase me for better results
        binary_operators=["*", "+", "-","/","^"],
        unary_operators=[
            'exp',
            'cos',
            'sin',
            'log',
            #'atan'
        ],
        constraints={'^': (-1, 1)},
        nested_constraints={"exp": {"exp": 0},
                            "^": {"^": 0},
                            "log": {"log": 0},
                            #"atan": {"cos": 1,"sin": 1, "exp": -1, "atan": 0},
                            "cos": {"cos": 0,"sin": 0},
                            "sin": {"cos": 0,"sin": 0}
                           },
        populations=36,
        verbosity=0,
        batching=True,
        warm_start = False
    )

In [ ]:
x_arr = np.array(x_list)
#for i in range(7):
i = 5
x = x_arr[:,i]
print('X_i',i)
input_values = np.expand_dims(np.array(mu_list).flatten(),-1)
target_values = np.expand_dims(x.flatten(),-1)
model.fit(input_values, target_values, 
        variable_names=["mu"])

In [ ]:
model

In [ ]:

from sympy.utilities.lambdify import lambdastr
from sympy.abc import a, m
N=None

test_preds = model.predict(input_values,N)
test_trues = target_values.reshape(test_preds.shape)
MSE = np.square(np.subtract(test_trues,test_preds)).mean() 
s = model.sympy(N).simplify()
display(s)
RMSE = np.sqrt(MSE)
print(RMSE)

print('Python Code:')
f = lambdastr((m), s)
print(f)

plt.scatter(input_values, test_trues)
plt.plot(input_values, test_preds,'--')
plt.grid()
plt.xlabel('mu')
plt.ylabel('X_'+str(i))
#plt.legend()
plt.show()

In [ ]:
def v2_fun(a, mu):

    x0 = (-0.09814221*mu - 0.0038020632*np.cos(57557.12*mu**3)) #7.1e-5
    x1 = (-mu**0.656669 + mu + 3.38790549618544e+15*mu**10.854148 - 0.0058048368*np.sin(0.39640546/mu) + 0.8174978) #9.2e-4
    x2 = ((0.00609094450050564 - 0.13287131*mu)/(mu - 0.057952844)) #3.4e-4
    x3 = ((0.04188588*mu - 0.010502198931628)/(mu - 0.0912081)) #5.8e-4
    x4 = (993.981895238186*mu**3 - 28.8640732668633*mu**2 - 0.3974241*mu - 0.0153266519781973*mu/math.sin(mu**(-0.8583719)) + 0.12678103) #1.5e-4
    x5 = (0.41461778 + 1.0344037/(mu*(math.cos(1.0518135/mu) + math.cos(math.exp(mu)/mu) + 125.503845) - 8.575114)) #2.1e-4
    x6 = (-mu**2/math.sin(math.log(mu) - 0.0911465258541601) + 0.07548606*mu + 0.0021057916) #7.3e-5

    y = (a + x0*(x1*a)**a)*(a*(x2*a + x3*np.sin(a) + x4*np.cos(a) + x5) + x6)/a #???
    return y

In [ ]:
import re
from typing import Dict, List, Tuple
import sympy as sp

def expr_to_latex_with_consts(
    expr_str: str,
    lhs: str = "v_1",
    litera: str = "C",
    j: int = 1,
    ignore_integer_floats: bool = True,
    functions: Dict[str, object] = None,
) -> Tuple[str, List[Tuple[int, float]]]:
    """
    Преобразует Python/NumPy выражение в LaTeX и заменяет константы на C_i^{j}.
    Возвращает (latex_string, список_констант [(i, value), ...]).
    """

    # 1) Отделим комментарий (например, "#2.7e-03") — если нужен RMSE, берите его заранее
    expr_no_comment = expr_str.split("#", 1)[0].strip()

    # 2) Поддержка np./math. — уберём префиксы
    expr_no_comment = re.sub(r"\b(np|math).", "", expr_no_comment)

    # 3) Базовые функции для sympy
    if functions is None:
        functions = {
            "exp": sp.exp,
            "cos": sp.cos,
            "sin": sp.sin,
            "tan": sp.tan,
            "sqrt": sp.sqrt,
            "log": sp.log,
        }

    # 4) Парсим выражение в sympy
    expr = sp.sympify(expr_no_comment, locals=functions)

    # 5) Соберём порядок появления дробных констант (по модулю) из исходной строки
    float_pattern = re.compile(r"(?<![w.])[-+]?d+.d+(?:[eE][-+]?d+)?")
    seen = []
    for m in float_pattern.finditer(expr_no_comment):
        v = float(m.group(0))
        if ignore_integer_floats and float(v).is_integer():
            continue
        key = abs(v)
        # Пропустим дубликаты по модулю
        if not any(abs(key - k) < 0 for k in seen):
            seen.append(key)

    # 6) Построим отображение |число| -> C_i
    absval_to_sym: Dict[float, sp.Symbol] = {}
    const_list: List[Tuple[int, float]] = []
    for i, val in enumerate(seen, start=1):
        sym = sp.Symbol(f"C{i}", real=True)
        absval_to_sym[val] = sym
        const_list.append((i, val))

    # 7) Функция замены чисел в sympy-дереве
    def replace_float(node):
        if isinstance(node, sp.Float):
            v = float(node)
            if ignore_integer_floats and float(v).is_integer():
                return node  # оставить как есть (например, 1.0, 2.0)
            key = abs(v)
            # Если встретили число, которого не было в "seen" (редко, но бывает из-за парсинга),
            # то добавим его в конец с новым индексом:
            if key not in absval_to_sym:
                sym = sp.Symbol(f"C{len(absval_to_sym) + 1}", real=True)
                absval_to_sym[key] = sym
                const_list.append((len(const_list) + 1, key))
            sym = absval_to_sym[key]
            return sym if v >= 0 else -sym
        return node

    expr_repl = expr.replace(lambda x: isinstance(x, sp.Float), replace_float)

    # 8) Получаем LaTeX
    latex_rhs = sp.latex(expr_repl)

    # 9) Превратим C1 -> C_{1}^{j}

    # Добавляем верхний индекс j к каждому C_{i}, если его ещё нет
    latex_rhs = re.sub(
        r"C_\{(\d)\}", 
        lambda m: f"{litera}_{{{m.group(1)}}}^{{{j}}}", 
        latex_rhs
    )

    # 10) Соберём полную строку с левой частью
    latex_full = f"{lhs} = {latex_rhs}"
    return latex_full, const_list

# ===== Пример использования =====
expr_str = "a*(-6.557952*mu + 0.26311928*np.exp(1.4665749*a) + 0.415885222275072)*np.exp(-1.4665749*a)*np.cos(0.5028553*a*mu - a + 0.93205124)"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_1", j=1)

print(latex)
print("Константы:")
for i, v in consts:
    print(f"C_{i}^{{1}} = {v}")

In [ ]:
def v1_fun(a, mu):
    y = (-a*((mu*np.cos(0.22937126**a - np.log(a)) - 0.41244736*np.cos(0.22130254**a + np.log(a)))*np.sin(a + 0.028886221) + 0.26014933*np.cos(1.2741468*a + 5.4069505))) #1.1e-03
    return y
def v2_fun(a, mu):
    y = (a*(-6.557952*mu + 0.26311928*np.exp(1.4665749*a) + 0.415885222275072)*np.exp(-1.4665749*a)*np.cos(0.5028553*a*mu - a + 0.93205124)) #2.7e-03
    #y = (((0.6295174*a - 3.436374)*(mu*(0.26111934*math.sin(a) + 1.22414) - 0.020510118)*math.cos(1.3607086*a - 1.0926718) + math.sin(1.2466944*a))/(3.436374 - 0.6295174*a)) #2.1e-03
    return y
def T_fun(a, mu):
    #y = (mu*(4.928138*math.cos(a - 0.6908796) + 2.9778968732458 + 43.95868/(math.exp(a) + math.sin(88.58396*mu)))*math.cos(math.exp(-83.87106*mu)) + 6.283895) #1.4e-02
    #y = mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + 45.56829193/(np.exp(a) + np.sin(87.19693384*mu)))*np.cos(np.exp(-65.93754813*mu)) + np.pi*2 #1.3e-02
    y = mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + (29/2)*np.pi/(np.exp(a) + np.sin(np.pi*(111/4)*mu)))*np.cos(np.exp(-21*np.pi*mu)) + np.pi*2 #1.3e-02
    return y

In [ ]:
from ConvertFormula import expr_to_latex_with_consts

In [ ]:
expr_str = "(-a*((mu*np.cos(0.22937126**a - np.log(a)) - 0.41244736*np.cos(0.22130254**a + np.log(a)))*np.sin(a + 0.028886221) + 0.26014933*np.cos(1.2741468*a + 5.4069505))) #1.1e-03"
litera = "B"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_x", j=6, litera=litera)

print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
expr_str = "(a*(-6.557952*mu + 0.26311928*np.exp(1.4665749*a) + 0.415885222275072)*np.exp(-1.4665749*a)*np.cos(0.5028553*a*mu - a + 0.93205124)) #2.7e-03"
litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=14, litera=litera)

print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
expr_str = "mu*(8*np.pi/5*np.cos(a - np.pi/5) + np.pi + (29/2)*np.pi/(np.exp(a) + np.sin(np.pi*(111/4)*mu)))*np.cos(np.exp(-21*np.pi*mu)) + np.pi*2 #1.3e-02"
litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=3, litera=litera)

print(latex)
print("Константы:")
for i, v in consts:
    print(f"{litera}_{i}^{{1}} = {v}")